In [0]:
from pyspark.sql import functions as F

CATALOG = "workspace"
dbutils.widgets.text("run_id", "manual")
dbutils.widgets.text("status", "success")
RUN_ID = dbutils.widgets.get("run_id")
STATUS = dbutils.widgets.get("status")
AUDIT = f"{CATALOG}.gold.pipeline_audit"

spark.sql(f"""CREATE TABLE IF NOT EXISTS {AUDIT} (
  run_id STRING, status STRING, recorded_at TIMESTAMP,
  bronze_orders BIGINT, silver_orders BIGINT, orders_quarantined BIGINT,
  fact_orders BIGINT, total_revenue DOUBLE, scd2_rows BIGINT, scd2_current BIGINT,
  cdc_quarantined BIGINT, reconciliation_ok BOOLEAN, notes STRING
) USING DELTA""")

def n(t):
    full = f"{CATALOG}.{t}"
    return spark.table(full).count() if spark.catalog.tableExists(full) else None

problems = []
bronze = silver = oq = fact = rev = scd_rows = scd_cur = cdcq = None
ok = None

if STATUS == "success":
    bronze, silver, oq = n("bronze.orders"), n("silver.orders"), n("silver.orders_quarantine")
    fact = n("gold.fact_orders")
    rev = float(spark.table(f"{CATALOG}.gold.fact_orders").agg(F.sum("total_amount")).first()[0])
    rev_sum = float(spark.table(f"{CATALOG}.gold.daily_sales_summary").agg(F.sum("revenue")).first()[0])
    scd = spark.table(f"{CATALOG}.gold.dim_customer_scd2")
    scd_rows = scd.count()
    scd_cur = scd.filter("is_current = true").count()
    dup_current = (scd.filter("is_current = true").groupBy("customer_id").count()
                      .filter("count > 1").count())
    cdcq = n("silver.customer_cdc_quarantine")

    if fact != silver:
        problems.append(f"fact rows ({fact}) differ from silver rows ({silver})")
    if abs(rev - rev_sum) > 0.01:
        problems.append(f"fact revenue ({rev}) differs from summary revenue ({rev_sum})")
    if bronze - oq < silver:
        problems.append("silver has more rows than bronze minus quarantined rows")
    if dup_current:
        problems.append(f"{dup_current} customers have more than one current SCD2 row")
    ok = len(problems) == 0
else:
    problems.append("a task in the job failed; see the job run for details")

schema = ("run_id string, status string, bronze_orders long, silver_orders long, "
          "orders_quarantined long, fact_orders long, total_revenue double, scd2_rows long, "
          "scd2_current long, cdc_quarantined long, reconciliation_ok boolean, notes string")
row = [(RUN_ID, STATUS, bronze, silver, oq, fact, rev, scd_rows, scd_cur, cdcq, ok,
        "; ".join(problems) if problems else "all checks passed")]

(spark.createDataFrame(row, schema)
    .withColumn("recorded_at", F.current_timestamp())
    .select("run_id", "status", "recorded_at", "bronze_orders", "silver_orders",
            "orders_quarantined", "fact_orders", "total_revenue", "scd2_rows",
            "scd2_current", "cdc_quarantined", "reconciliation_ok", "notes")
    .write.format("delta").mode("append").saveAsTable(AUDIT))

display(spark.table(AUDIT).orderBy(F.desc("recorded_at")).limit(5))

if STATUS == "success" and problems:
    raise Exception("Reconciliation failed: " + "; ".join(problems))

run_id,status,recorded_at,bronze_orders,silver_orders,orders_quarantined,fact_orders,total_revenue,scd2_rows,scd2_current,cdc_quarantined,reconciliation_ok,notes
manual,success,2026-10-29T11:09:50.440Z,15658,15501,3,15500,1.094411035E8,644,501,8,false,fact rows (15500) differ from silver rows (15501)


---------------------------------------------------------------------------
Exception                                 Traceback (most recent call last)
File <command-4658158558552329>, line 65
     62 display(spark.table(AUDIT).orderBy(F.desc("recorded_at")).limit(5))
     64 if STATUS == "success" and problems:
---> 65     raise Exception("Reconciliation failed: " + "; ".join(problems))

Exception: Reconciliation failed: fact rows (15500) differ from silver rows (15501)